# Development-session model selection

All family, feature, and hyperparameter choices occur here. The latest completed Fall/Winter and Summer sessions, `20259` and `20265`, are reserved for evaluation and are not loaded.

In [1]:
from __future__ import annotations
import hashlib, inspect, json, os, platform, sys
from pathlib import Path
import joblib, numpy as np, pandas as pd, sklearn
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import brier_score_loss, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT.name == 'v2' or not (PROJECT_ROOT / 'notebooks').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_ROOT = PROJECT_ROOT / 'data' / 'Enrollment-Data-master'
ARTIFACT_ROOT = PROJECT_ROOT / 'artifacts' / 'v2'
CACHE_ROOT = ARTIFACT_ROOT / 'cache'
MODEL_ROOT = PROJECT_ROOT / 'model'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
CACHE_ROOT.mkdir(parents=True, exist_ok=True)
RANDOM_STATE = 20260812
SESSION_ORDER = ['20229','20235','20239','20245','20249','20255','20259','20265']
SEASONS = {'fall_winter':['20229','20239','20249','20259'], 'summer':['20235','20245','20255','20265']}
FINAL_TEST = {'fall_winter':'20259', 'summer':'20265'}
DEVELOPMENT = {k:[s for s in v if s != FINAL_TEST[k]] for k,v in SEASONS.items()}

def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''): h.update(chunk)
    return h.hexdigest()

def fingerprint(payload) -> str:
    return hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:16]

def versions():
    return {'python':platform.python_version(),'numpy':np.__version__,'pandas':pd.__version__,
            'scikit_learn':sklearn.__version__,'joblib':joblib.__version__}
BASE = ['position_to_capacity','waitlist_to_capacity','days_to_deadline','movement_3d','movement_7d','position','waitlist','capacity','capacity_changed_7d','position_to_waitlist','days_squared','log_waitlist','movement_velocity_7d']
CONTEXT = BASE + ['near_deadline_7d','days_under_7','days_under_14','days_over_60','position_ratio_near_7d','waitlist_ratio_near_7d','rank_over_30pct','campus_erin','campus_scar','term_winter','term_full_year','winter_near_7d','scar_near_7d']
RANK_MONOTONIC={'position_to_capacity':-1,'position':-1,'position_to_waitlist':-1,'position_ratio_near_7d':-1,'rank_over_30pct':-1}
def targeted(frame):
    f=frame.copy(); days=f.days_to_deadline.astype(float); near=(days<=7).astype('float32')
    f['near_deadline_7d']=near; f['days_under_7']=(7-days).clip(lower=0); f['days_under_14']=(14-days).clip(lower=0); f['days_over_60']=(days-60).clip(lower=0)
    f['position_ratio_near_7d']=f.position_to_capacity*near; f['waitlist_ratio_near_7d']=f.waitlist_to_capacity*near
    f['rank_over_30pct']=(f.position_to_capacity>.30).astype('float32'); f['campus_erin']=(f.campus=='ERIN').astype('float32'); f['campus_scar']=(f.campus=='SCAR').astype('float32')
    f['term_winter']=(f.term=='winter').astype('float32'); f['term_full_year']=(f.term=='full_year').astype('float32')
    f['winter_near_7d']=f.term_winter*near; f['scar_near_7d']=f.campus_scar*near
    return f
def make_model(features, *, leaf=15, l2=3.0):
    constraints=[RANK_MONOTONIC.get(x,0) for x in features]
    return Pipeline([('features',ColumnTransformer([('numeric',SimpleImputer(strategy='median'),features)],remainder='drop')),
      ('model',HistGradientBoostingClassifier(max_iter=200,learning_rate=.05,max_leaf_nodes=leaf,l2_regularization=l2,monotonic_cst=constraints,random_state=RANDOM_STATE))])
def ece(y,p,w,bins=10):
    edges=np.linspace(0,1,bins+1); ids=np.clip(np.digitize(p,edges)-1,0,bins-1); total=w.sum(); out=0
    for b in range(bins):
        m=ids==b
        if m.any(): out+=w[m].sum()/total*abs(np.average(y[m],weights=w[m])-np.average(p[m],weights=w[m]))
    return float(out)
def metrics(frame,p):
    y=frame.cleared.to_numpy(); w=frame.model_weight.to_numpy(); p=np.clip(np.asarray(p),1e-6,1-1e-6)
    auc=roc_auc_score(y,p,sample_weight=w) if np.unique(y).size>1 else np.nan
    return {'brier':brier_score_loss(y,p,sample_weight=w),'log_loss':log_loss(y,p,sample_weight=w,labels=[0,1]),'ece':ece(y,p,w),'auc':auc,'accuracy':np.average((p>=.5)==y,weights=w)}


## Load development sessions only

In [2]:
manifest=json.loads((ARTIFACT_ROOT/'cache-manifest.json').read_text())
CACHE_VERSION=int(manifest['cache_version'])
session_samples={s:targeted(pd.read_pickle(CACHE_ROOT/f'{s}-positions-v{CACHE_VERSION}.pkl')) for sessions in DEVELOPMENT.values() for s in sessions}
assert not set(FINAL_TEST.values()).intersection(session_samples)
cache_fingerprint=fingerprint(manifest)
{s:len(f) for s,f in session_samples.items()}

{'20229': 297352,
 '20239': 298197,
 '20249': 300942,
 '20235': 143325,
 '20245': 172834,
 '20255': 151030}

## Nested rolling development folds

In [3]:
CANDIDATES={
 'boosted_base':{'features':BASE,'leaf':15,'l2':3.0},
 'boosted_context':{'features':CONTEXT,'leaf':15,'l2':3.0},
 'boosted_context_leaf31':{'features':CONTEXT,'leaf':31,'l2':3.0},
 'boosted_context_l2_1':{'features':CONTEXT,'leaf':15,'l2':1.0},
 'boosted_context_l2_10':{'features':CONTEXT,'leaf':15,'l2':10.0},
}
rows=[]; prediction_rows=[]; total_fits=sum((len(sessions)-1)*len(CANDIDATES) for sessions in DEVELOPMENT.values()); fit_number=0
for season,sessions in DEVELOPMENT.items():
    for fold in range(1,len(sessions)):
        train=pd.concat([session_samples[s] for s in sessions[:fold]],ignore_index=True); valid=session_samples[sessions[fold]]
        for name,spec in CANDIDATES.items():
            fit_number+=1; print(f'[{fit_number}/{total_fits}] {season} {sessions[fold]} {name}: fitting {len(train):,} rows',flush=True)
            model=make_model(**spec); model.fit(train[spec['features']],train.cleared,model__sample_weight=train.model_weight)
            probability=model.predict_proba(valid[spec['features']])[:,1]
            rows.append({'season':season,'valid_session':sessions[fold],'candidate':name,**metrics(valid,probability)})
            prediction_rows.append(pd.DataFrame({'season':season,'valid_session':sessions[fold],'offering_id':valid.offering_id.to_numpy(),'row_id':valid.index.to_numpy(),'candidate':name,'y':valid.cleared.to_numpy(),'w':valid.model_weight.to_numpy(),'p':probability}))
            print(f'[{fit_number}/{total_fits}] complete: Brier {rows[-1]["brier"]:.6f}',flush=True)
scores=pd.DataFrame(rows); development_predictions=pd.concat(prediction_rows,ignore_index=True); scores

[1/20] fall_winter 20239 boosted_base: fitting 297,352 rows


[1/20] complete: Brier 0.113495


[2/20] fall_winter 20239 boosted_context: fitting 297,352 rows


[2/20] complete: Brier 0.108586


[3/20] fall_winter 20239 boosted_context_leaf31: fitting 297,352 rows


...

[19/20] summer 20255 boosted_context_l2_1: fitting 316,159 rows


[19/20] complete: Brier 0.006886


[20/20] summer 20255 boosted_context_l2_10: fitting 316,159 rows


[20/20] complete: Brier 0.004281


,season,valid_session,candidate,brier,log_loss,ece,auc,accuracy
0,fall_winter,20239,boosted_base,0.113495,0.365888,0.024281,0.778077,0.856104
1,fall_winter,20239,boosted_context,0.108586,0.343620,0.022741,0.816584,0.857830
2,fall_winter,20239,boosted_context_leaf31,0.108643,0.344614,0.023790,0.816877,0.857262
3,fall_winter,20239,boosted_context_l2_1,0.108704,0.343751,0.022310,0.816043,0.857532
4,fall_winter,20239,boosted_context_l2_10,0.108717,0.344965,0.023848,0.815581,0.858640
5,fall_winter,20249,boosted_base,0.045701,0.193221,0.091298,0.761431,0.963593
6,fall_winter,20249,boosted_context,0.044968,0.182865,0.086552,0.812153,0.964236
7,fall_winter,20249,boosted_context_leaf31,0.045683,0.182465,0.085812,0.815583,0.963208
8,fall_winter,20249,boosted_context_l2_1,0.045077,0.183042,0.086464,0.810837,0.964347
9,fall_winter,20249,boosted_context_l2_10,0.045108,0.183443,0.086752,0.809187,0.964386


## Stable selection and fingerprinted checkpoint

In [4]:
summary=scores.groupby('candidate').agg(mean_brier=('brier','mean'),worst_brier=('brier','max'),mean_ece=('ece','mean'),mean_auc=('auc','mean')).sort_values(['mean_brier','worst_brier'])
def paired_candidate_ci(candidate, reference='boosted_base', reps=1000):
    wide=development_predictions.loc[development_predictions.candidate.isin([candidate,reference])].pivot(index=['season','valid_session','offering_id','row_id','y','w'],columns='candidate',values='p').reset_index()
    # Reduce each offering to weighted squared-error sums once. Bootstrap only these small arrays.
    wide['_candidate_error']=wide.w*(wide.y-wide[candidate])**2; wide['_reference_error']=wide.w*(wide.y-wide[reference])**2
    offering=wide.groupby(['season','valid_session','offering_id'],sort=True).agg(candidate_error=('_candidate_error','sum'),reference_error=('_reference_error','sum'),weight=('w','sum')).reset_index()
    rng=np.random.default_rng(RANDOM_STATE); differences=np.empty(reps); groups=[frame for _,frame in offering.groupby(['season','valid_session'],sort=True)]
    for repetition in range(reps):
        fold_differences=[]
        for frame in groups:
            chosen=rng.integers(0,len(frame),len(frame)); weight=frame.weight.to_numpy()[chosen].sum()
            fold_differences.append((frame.candidate_error.to_numpy()[chosen].sum()-frame.reference_error.to_numpy()[chosen].sum())/weight)
        differences[repetition]=np.mean(fold_differences)
    return np.quantile(differences,[.025,.5,.975]).tolist()
paired={name:paired_candidate_ci(name) for name in CANDIDATES if name!='boosted_base'}
point_winner=summary.index[0]
winner=point_winner if point_winner=='boosted_base' or paired[point_winner][2]<0 else 'boosted_base'
locked={'candidate':winner,'point_winner':point_winner,'spec':CANDIDATES[winner],'development_sessions':DEVELOPMENT,
 'cache_fingerprint':cache_fingerprint,'versions':versions(),'scores':scores.to_dict('records'),'paired_brier_differences_vs_base':paired}
locked['fingerprint']=fingerprint(locked)
for old_path in ARTIFACT_ROOT.glob('development-selection-*.json'): old_path.unlink()
path=ARTIFACT_ROOT/f'development-selection-{locked["fingerprint"]}.json'
path.write_text(json.dumps(locked,indent=2),encoding='utf-8')
summary, paired, winner, path

(                        mean_brier  worst_brier  mean_ece  mean_auc
 candidate                                                          
 boosted_context_l2_10     0.282085     0.970236  0.273999  0.708256
 boosted_context           0.282355     0.970236  0.273552  0.709579
 boosted_base              0.282547     0.970236  0.275110  0.679836
 boosted_context_leaf31    0.282563     0.970236  0.273591  0.710820
 boosted_context_l2_1      0.282726     0.970236  0.273437  0.708960,
 {'boosted_context': [-0.0009066468129213068,
   -0.00020909746551100354,
   0.0006297663177565265],
  'boosted_context_leaf31': [-0.0007681220096623293,
   -1.4877128240118232e-05,
   0.000853628660006441],
  'boosted_context_l2_1': [-0.0007094378402463557,
   0.00014412988381343336,
   0.001172322658865667],
  'boosted_context_l2_10': [-0.0009948668191083992,
   -0.00046412048828687013,
   0.00013484370210448007]},
 'boosted_base',
 PosixPath('/student/anfazsha/oracle/artifacts/v2/development-selection-2e6193

## Decision rule

The chosen specification is locked by its data, parameter, result, library-version fingerprint, and paired offering-clustered comparison. Notebook 3 must consume this exact fingerprint, and the latest-session evaluation results may not influence this choice.